### Selecting best/optimal variables for your dataset, new tools

In [14]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("winequality-red.csv")

### RFE - recursive feature elimination

In [15]:
# typical X/y -split
X = df.drop("quality", axis=1)
y = df['quality']

# define model (linear regression, random forest, XGBoost etc.)
# technically you can use pretty much any classic ML algorithm
# model = LinearRegression()

# idea: you might want to try RFE with multuple ML algorithms
# and then cross-validate the results (which results are common in all models)

# another idea: if you plan on using e.g. XGBoost in your final model
# it's probably a good idea to use the same XGBoost here
model = RandomForestRegressor()

# create RFE, place the model and choose number of optimal variables
rfe = RFE(estimator=model, n_features_to_select=7)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and results
rankings = rfe.ranking_
support = rfe.support_

# build a DataFrame to wrap up result for easier inspection
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# you can use these results with any other knowledge you have
# from other optimal variable selection tools, and cross-validation
results_df

,Feature,Ranking,Selected
1,volatile acidity,1,True
7,density,1,True
6,total sulfur dioxide,1,True
4,chlorides,1,True
9,sulphates,1,True
10,alcohol,1,True
8,pH,1,True
3,residual sugar,2,False
5,free sulfur dioxide,3,False
0,fixed acidity,4,False


In [16]:
# the selection loosely follow the correlations it seems
# this might imply the data is more or less linear or something else
df.corr()['quality'].sort_values(ascending=False)

quality                 1.000000
alcohol                 0.476166
sulphates               0.251397
citric acid             0.226373
fixed acidity           0.124052
residual sugar          0.013732
free sulfur dioxide    -0.050656
pH                     -0.057731
chlorides              -0.128907
density                -0.174919
total sulfur dioxide   -0.185100
volatile acidity       -0.390558
Name: quality, dtype: float64

### Mutual information - another alternative

In [17]:
# fit the mutual information algorithm
# red wine dataset can be used for both regression and classification
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

alcohol                 0.175970
volatile acidity        0.151875
sulphates               0.097726
total sulfur dioxide    0.085964
density                 0.074471
fixed acidity           0.060556
free sulfur dioxide     0.056602
citric acid             0.048176
residual sugar          0.035866
chlorides               0.030421
pH                      0.011594
dtype: float64

### Mutual information - classification alternative

In [18]:
# fit the mutual information algorithm
# red wine dataset can be used for both regression and classification
# => that's why either or works
mi = mutual_info_classif(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

alcohol                 0.167987
volatile acidity        0.119042
sulphates               0.101513
density                 0.087740
total sulfur dioxide    0.079257
citric acid             0.072422
fixed acidity           0.037798
pH                      0.029334
chlorides               0.025412
free sulfur dioxide     0.020310
residual sugar          0.018371
dtype: float64